In [63]:
import pandas as pd
import numpy as np
import time as time_lib
import warnings
warnings.filterwarnings("ignore")

In [64]:
# Create n-by-m matrix (input: df, output: new_df)
def create_data(df, n, m):
  
  pivot_df = df.pivot_table(index="item_id", columns="user_id",values="rating", aggfunc='mean')
  
  n_row = pivot_df.shape[0]
  n_col = pivot_df.shape[1]
  Obs_ind = np.where(pivot_df.notnull())    # Row and column indices for the observed entries of "Mdat"
  num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
  sparsity = 1 - num_Obs / (n_row * n_col)
  #print(n_row,n_col,num_Obs,sparsity)
  print(f'(Original) matrix sparsity: {sparsity:f}')

  #Sorting and extracting 
  new_df1 = pivot_df.copy()
  new_df1['count'] = new_df1.count(axis=1)
  new_df1_sort = new_df1.sort_values('count', ascending=False)
  new_df2 = new_df1_sort.drop("count", axis=1, inplace=False)
  new_df3 = new_df2.transpose()
  new_df3['count'] = new_df3.count(axis=1)
  new_df3_sort = new_df3.sort_values('count', ascending=False)  
  final_df = new_df3_sort.drop("count", axis=1, inplace=False)
  final_data = final_df.transpose()
  new_df = final_data.iloc[0:n,0:m]
  #print(new_df)
  
  n_row = new_df.shape[0]
  n_col = new_df.shape[1]
  Obs_ind = np.where(new_df.notnull())    # Row and column indices for the observed entries of "Mdat"
  num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
  sparsity = 1 - num_Obs / (n_row * n_col)
  #print(n_row,n_col,num_Obs,sparsity)
  sparsity = 1 - num_Obs / (m * n)

  print(f'(Extracted) matrix sparsity: {sparsity:f}')


  return new_df

In [65]:
#Create the k-fold datasets 
import random
def k_fold_data(df,k,random_seed):

  n_row = df.shape[0]
  n_col = df.shape[1]
  Obs_ind = np.where(df.notnull())    # Row and column indices for the observed entries of "Mdat"
  num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
  sparsity = 1 - num_Obs / (n_row * n_col)
  #print(n_row,n_col,num_Obs,sparsity)
  print(f'(Original) matrix sparsity: {sparsity:f}')

  a = int(num_Obs/k)
  A = k*a + k - num_Obs
  num_Obs_each = np.append(a*np.ones(A), (a+1)*np.ones(k-A)).astype(int)
  #print(num_Obs_each)

  label = np.array(range(num_Obs))

  p = [None] * k
  test_dat = [None] * k

  for i in range(k):

    Tdat = pd.DataFrame.copy(df)
    random.seed(random_seed)
    random.shuffle(label)
    p[i] = label[:num_Obs_each[i]]
    #print(p[i])
    for j in p[i]:
      Tdat.iloc[Obs_ind[0][j], Obs_ind[1][j]] = np.nan
    
    test_dat[i] = Tdat
    #print(test_dat[i])
    #print(len(np.where(test_dat[i].notnull())[0]))
    sparsity = 1 -  len(np.where(test_dat[i].notnull())[0]) / (test_dat[i].shape[0] * test_dat[i].shape[1])
    print(str(i)+"th data sparsity = "+str(sparsity))

  return test_dat


In [66]:


#main loop 

k = 10 # k-fold validation 
item_set = [10]
user_set = [10]


#prepare the result summary table 
result_summary_df = pd.DataFrame(columns = ['items','users','sparsity','avg_acc','avg_rmse','avg_mad','avg_time'])

#main 

for n in item_set: 
    for m in user_set:
        n_items = n 
        n_users = m 

        #create the datasets 
        random_seed = 20230530 
        # from google.colab import files
        url = "https://raw.githubusercontent.com/park61/imputation/main/u.data"
        
        colnames = ['user_id', 'item_id', 'rating', 'timestamp']
        df = pd.read_csv(url,sep='\t', names=colnames, header=None)
        new_df = create_data(df, n_items, n_users)
        new_df.to_csv('./data/'+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv')

        test_dat = k_fold_data(new_df,k,random_seed)

        #Checking invalid dataset 
        for i in range(k):
            print(i, min((1-test_dat[i].isna()).sum(axis=0)), min((1-test_dat[i].isna()).sum(axis=1)))

        for i in range(k):
            if i<9:
                filename = './data/'+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(i+1)+'.csv'
            else:
                filename = './data/'+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(i+1)+'.csv'
            test_dat[i].to_csv(filename)

        ten = [str(j).zfill(2) for j in list(range(1,k+1))]
        print(ten)
        
        labs = ['test'+i for i in ten]
        labs.insert(0,'orig')
        
        url_dict = {}
        url_dict["orig"] = './data/'+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
        
        for i in ten:
            url_dict["test{0}".format(i)] = './data/'+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'
        print(labs)
        
        for lab in labs:
            print(url_dict[lab])    
        
        df_dict = {}
        for lab in labs:
            #print(lab)
            df = pd.read_csv(url_dict[lab])
            #print(lab)
            df_dict[lab] = df.set_index('item_id')

        #compute the sparsity 
        n_row = df_dict["orig"].shape[0]
        n_col = df_dict["orig"].shape[1]
        Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
        num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
        sparsity = 1 - num_Obs / (n_row * n_col)
        print("sparsity: "+str(sparsity))

        #Run algorithm 
        acc_list = []
        rmse_list = []
        mad_list = []
        time_list = []
        count_list = []

        df_orig = df_dict["orig"]
        df_orig.columns = range(df_orig.shape[1])

        mm = df_orig.shape[0]
        nn = df_orig.shape[1]

        labs_test = labs[1:]

        col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
        count = 0 

        for lab in labs_test:
            count += 1
            #print(lab)
            
            df = df_dict[lab]
            
            masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
            #df, maxs = normal1(masked_df)
            #df, maxs = normal2(masked_df)
            #df, maxs, probs = normal3(masked_df)

            # from missingpy import MissForest
            # imputer = MissForest()

            from fancyimpute import SoftImpute
            imputer = SoftImpute(verbose=False)

            start = time_lib.time()
            imputed = pd.DataFrame(imputer.fit_transform(df))

            # Rounding for when no normalization           
            for i in range(mm):
                for j in range(nn):
                    x = imputed.iloc[i,j]
                    if x < 1:
                        imputed.iloc[i,j] = 1
                    elif x > col_max[j]:
                        imputed.iloc[i,j] = col_max[j]
                    else:
                        imputed.iloc[i,j] = round(x,0)

            #imputed = unnormal1(imputed, maxs)
            #imputed = unnormal2(imputed, maxs)
            #imputed = unnormal3(imputed, maxs, probs)

            end = time_lib.time()
            time = end - start

            df_orig.index = range(mm)
            imputed.index = range(mm)

            df_orig.columns = range(nn)
            imputed.columns = range(nn)

            #save the result data
            url = './result/'

            if count<10:
                filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_soft_imputed.csv'
            else:
                filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_soft_imputed.csv'
            
            imputed.to_csv(filename)
            print("saved the result file as "+str(filename))
            diff = df_orig - imputed
            sq_diff = diff ** 2
            abs_diff = abs(diff)

            n_match = 0
            for i in range(mm):
                for j in range(nn):
                    if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
                        n_match += 1
            acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
            rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
            mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
            acc_list.append(acc)
            rmse_list.append(rmse)
            mad_list.append(mad)
            time_list.append(time)
            count_list.append(count)
        df = pd.DataFrame(data=[ acc_list, rmse_list,mad_list,time_list])
        display(df.T)
        df_temp = df.T
        filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data_all_soft_imputed.csv'
        df_temp.to_csv(filename)
        print('saved'+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+ 'the summary result file as' +str(filename))
        # 열별 평균값 계산
        mean_values = df_temp.mean()
        # 열별 표준편차 계산
        std_values = df_temp.std()
        new_row = [n,m,sparsity]+mean_values.tolist()
        display(new_row)
        result_summary_df = result_summary_df.append(pd.Series(new_row, index=result_summary_df.columns), ignore_index=True)
        filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'soft_imputed.csv'
        df_temp.to_csv(filename)
        print("saved the summary result file as "+str(filename))


display(result_summary_df)

        

        

(Original) matrix sparsity: 0.936953
(Extracted) matrix sparsity: 0.110000
(Original) matrix sparsity: 0.110000
0th data sparsity = 0.18999999999999995
1th data sparsity = 0.19999999999999996
2th data sparsity = 0.19999999999999996
3th data sparsity = 0.19999999999999996
4th data sparsity = 0.19999999999999996
5th data sparsity = 0.19999999999999996
6th data sparsity = 0.19999999999999996
7th data sparsity = 0.19999999999999996
8th data sparsity = 0.19999999999999996
9th data sparsity = 0.19999999999999996
0 3 6
1 3 5
2 2 7
3 3 6
4 3 6
5 3 6
6 3 6
7 3 5
8 3 7
9 2 6
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10']
['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
./data/10-by-10_original_matrix.csv
./data/20230530_10-by-10_10_fold_test_data01.csv
./data/20230530_10-by-10_10_fold_test_data02.csv
./data/20230530_10-by-10_10_fold_test_data03.csv
./data/20230530_10-by-10_10_fold_test_data04.csv
./data/20230530_10-by-10_1

,0,1,2,3
0,0.000000,1.541104,1.375000,0.015512
1,0.555556,1.598611,1.000000,0.016566
2,0.444444,1.247219,0.888889,0.015420
3,0.111111,1.563472,1.333333,0.013386
4,0.444444,1.105542,0.777778,0.013730
5,0.444444,0.942809,0.666667,0.013525
6,0.333333,1.154701,0.888889,0.012319
7,0.555556,0.881917,0.555556,0.013851
8,0.333333,1.374369,1.000000,0.015973
9,0.000000,1.290994,1.222222,0.015377


saved20230530_10-by-10the summary result file as./result/20230530_10-by-10_10_fold_test_data_all_soft_imputed.csv


[10,
 10,
 0.10999999999999999,
 0.3222222222222223,
 1.2700736328421947,
 0.9708333333333332,
 0.01456589698791504]

saved the summary result file as ./result/20230530_10-by-10_soft_imputed.csv


,items,users,sparsity,avg_acc,avg_rmse,avg_mad,avg_time
0,10.0,10.0,0.11,0.322222,1.270074,0.970833,0.014566


In [67]:
df_result_all = pd.DataFrame(columns = ['items','users','sparsity','avg_acc','avg_rmse','avg_mad','avg_time'])
display(df_result_all)
# 데이터프레임 생성
df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 10]})
display(df)
# 열별 평균값 계산
mean_values = df.mean()
display(mean_values)
# 열별 표준편차 계산
std_values = df.std()
display(std_values)



,items,users,sparsity,avg_acc,avg_rmse,avg_mad,avg_time


,A,B
0,1,4
1,2,5
2,3,10


A    2.000000
B    6.333333
dtype: float64

A    1.00000
B    3.21455
dtype: float64

In [68]:
mean_values.tolist()

[2.0, 6.333333333333333]

In [69]:
new_row = ['a','b','c']+mean_values.tolist()
new_row

['a', 'b', 'c', 2.0, 6.333333333333333]

In [70]:
df = pd.DataFrame(mean_values).T
display(df)
display(df.iloc[0].values)
new_row = ['a','b','c']+ df.iloc[0].values.tolist()#, df.iloc[0].values]
new_row

,A,B
0,2.0,6.333333


array([2.        , 6.33333333])

['a', 'b', 'c', 2.0, 6.333333333333333]

In [71]:
import pandas as pd
import numpy as np
n_items = 100
n_users = 300
k = 10 
ten = [str(j).zfill(2) for j in list(range(1,k+1))]
ten

['01', '02', '03', '04', '05', '06', '07', '08', '09', '10']

In [72]:
labs = ['test'+i for i in ten]
labs.insert(0,'orig')
url = ""#"https://raw.githubusercontent.com/park61/imputation/main/data/"
url_dict = {}
url_dict["orig"] = url+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
for i in ten:
    url_dict["test{0}".format(i)] = url+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'

In [73]:
print(labs)
for lab in labs:
  print(url_dict[lab])

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
100-by-300_original_matrix.csv
100-by-300_10_fold_test_data01.csv
100-by-300_10_fold_test_data02.csv
100-by-300_10_fold_test_data03.csv
100-by-300_10_fold_test_data04.csv
100-by-300_10_fold_test_data05.csv
100-by-300_10_fold_test_data06.csv
100-by-300_10_fold_test_data07.csv
100-by-300_10_fold_test_data08.csv
100-by-300_10_fold_test_data09.csv
100-by-300_10_fold_test_data10.csv


In [74]:
df_dict = {}
for lab in labs:
  print(lab)
  df = pd.read_csv(url_dict[lab])
  print(lab)
  df_dict[lab] = df.set_index('item_id')

orig


FileNotFoundError: [Errno 2] No such file or directory: '100-by-300_original_matrix.csv'

In [ ]:
n_row = df_dict["orig"].shape[0]
n_col = df_dict["orig"].shape[1]
Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
sparsity = 1 - num_Obs / (n_row * n_col)
print(sparsity)

In [ ]:
#@title Run SoftImpute algorithm
import time as time_lib
import warnings
warnings.filterwarnings("ignore")

acc_list = []
rmse_list = []
mad_list = []
time_list = []

df_orig = df_dict["orig"]
df_orig.columns = range(df_orig.shape[1])

mm = df_orig.shape[0]
nn = df_orig.shape[1]

labs_test = labs[1:]

col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
count = 0 

for lab in labs_test:
  count += 1
  print(lab)
  
  df = df_dict[lab]
  import numpy as np
  masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
  #df, maxs = normal1(masked_df)
  #df, maxs = normal2(masked_df)
  #df, maxs, probs = normal3(masked_df)

  # from missingpy import MissForest
  # imputer = MissForest()

  from fancyimpute import SoftImpute
  imputer = SoftImpute(verbose=False)

  start = time_lib.time()
  imputed = pd.DataFrame(imputer.fit_transform(df))

  # Rounding for when no normalization           
  for i in range(mm):
    for j in range(nn):
      x = imputed.iloc[i,j]
      if x < 1:
        imputed.iloc[i,j] = 1
      elif x > col_max[j]:
        imputed.iloc[i,j] = col_max[j]
      else:
        imputed.iloc[i,j] = round(x,0)

  #imputed = unnormal1(imputed, maxs)
  #imputed = unnormal2(imputed, maxs)
  #imputed = unnormal3(imputed, maxs, probs)

  end = time_lib.time()
  time = end - start

  df_orig.index = range(mm)
  imputed.index = range(mm)

  df_orig.columns = range(nn)
  imputed.columns = range(nn)

  #save the result data
  
  if count<10:
    filename = str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_soft_imputed.csv'
  else:
    filename = str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_soft_imputed.csv'
    
  #imputed.to_csv(filename)
  diff = df_orig - imputed
  sq_diff = diff ** 2
  abs_diff = abs(diff)

  n_match = 0
  for i in range(mm):
    for j in range(nn):
      if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
        n_match += 1
  acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
  print(acc)  
  rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
  mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
  print(rmse)
  print(mad)
  print(time)
  acc_list.append(acc)
  rmse_list.append(rmse)
  mad_list.append(mad)
  time_list.append(time)

In [ ]:
print(sparsity)

print("Accuracy")
for i in acc_list:
  print(i)
print("\nRMSE")
for i in rmse_list:
  print(i)
print("\nMAD")
for i in mad_list:
  print(i)
print("\nTime")
for i in time_list:
  print(i)

In [ ]:
df = pd.DataFrame(data=[acc_list, rmse_list,mad_list,time_list])
df.T